In [1]:
import os
import re
import requests
import pandas as pd
from base64 import b64decode
from dotenv import load_dotenv
from time import sleep

# === Load tokens ===
load_dotenv("All_Tokens.env")
tokens = [os.getenv(f"GITHUB_TOKEN_{i}") for i in range(1, 7)]
tokens = [t for t in tokens if t]
if not tokens:
    raise ValueError("❌ No GitHub tokens found in All_Tokens.env")
token_index = 0

def get_headers():
    global token_index
    token = tokens[token_index]
    token_index = (token_index + 1) % len(tokens)
    return {"Authorization": f"token {token}"}

# === Define keyword baskets ===
keywords = [
    'example', 'sample', 'demo', 'test', 'debug', 'presentation', 'module', 'components',
    'lib', 'library', 'sdk', 'utils', 'utility', 'plugin', 'widget', 'playground', 'framework',
    'architecture', 'skeleton', 'collection', 'starting point', 'protocol', 'benchmark', 'hackathon',
    'classroom', 'course', 'exercise', 'assignment', 'homework', 'assessment', 'interview', 'asset',
    'template', 'catalog', 'tutorial', 'tool'
]
preceded_by = ['this', 'is a', 'is an', 'our', 'my']
not_preceded_by = ['using', 'with']

# === Paths ===
input_path = r"C:\Android Mobile App\Step1_URL_Search\Type_2_Searching_Pipeline\step4_manifest_check_output.csv"
output_path = r"C:\Android Mobile App\Step1_URL_Search\Type_2_Searching_Pipeline\step5_removal_keyword_check_output.csv"
interim_path = output_path.replace(".csv", "_interim.csv")

# === Load data ===
if os.path.exists(interim_path):
    print(f"🔄 Resuming from interim: {interim_path}")
    df = pd.read_csv(interim_path)
else:
    df = pd.read_csv(input_path)
    df["removal_keyword_flag"] = "none"
    df["removal_reason"] = "none"
    df["Valid_Repo_Step5"] = "none"

# === Filter unreviewed repos ===
to_review = df[(df["Valid_Repo_Step4"] == "yes") & (df["Valid_Repo_Step5"] == "none")]
print(f"🔍 Reviewing {len(to_review)} repos...\n")

# === Helper functions ===
def fetch_manifest_paths(repo):
    url = f"https://api.github.com/search/code?q=filename:AndroidManifest.xml+repo:{repo}"
    r = requests.get(url, headers=get_headers())
    if r.status_code == 200:
        return [item["path"] for item in r.json().get("items", [])]
    return []

def fetch_readme(repo):
    url = f"https://api.github.com/repos/{repo}/readme"
    r = requests.get(url, headers=get_headers())
    if r.status_code == 200:
        try:
            content = r.json().get("content", "")
            return b64decode(content).decode('utf-8', errors='ignore')
        except:
            return ""
    return ""

def has_removal_context(text, k):
    # Must match keyword and be preceded by a context phrase but not negated
    if re.search(rf'(?<!\S){re.escape(k)}(?!\S)', text, re.IGNORECASE):
        if any(re.search(rf'(?<!\S){re.escape(w)}\s+(\S+\s+){{0,4}}{re.escape(k)}(?!\S)', text, re.IGNORECASE) for w in preceded_by):
            if not any(re.search(rf'{re.escape(n)}\s+(\S+\s+){{0,4}}{re.escape(k)}', text, re.IGNORECASE) for n in not_preceded_by):
                return True
    return False

# === Main loop ===
for i, idx in enumerate(to_review.index, 1):
    row = df.loc[idx]
    repo = row["full_name"]
    name = row["name"].lower()
    topics = str(row["topics"]).lower()
    description = str(row.get("description", "")).lower()

    print(f"[{i}/{len(to_review)}] 🔍 Checking: {repo}")

    reasons = []

    try:
        manifest_paths = fetch_manifest_paths(repo)
        readme = fetch_readme(repo)

        # 1. Check manifest paths
        if any(k in path.lower() for k in keywords for path in manifest_paths):
            reasons.append("manifest_path")

        # 2. Check repo name and topics
        if any(k in name for k in keywords) or any(k in topics for k in keywords):
            reasons.append("repo_name")

        # 3. Check description with context
        if any(has_removal_context(description, k) for k in keywords):
            reasons.append("description")

        # 4. Check README with context
        if any(has_removal_context(readme, k) for k in keywords):
            reasons.append("readme")

        flag = "yes" if reasons else "no"
        df.at[idx, "removal_keyword_flag"] = flag
        df.at[idx, "Valid_Repo_Step5"] = "no" if flag == "yes" else "yes"
        df.at[idx, "removal_reason"] = ", ".join(reasons) if reasons else "none"

    except Exception as e:
        print(f"❌ Error with {repo}: {e}")
        df.at[idx, "removal_keyword_flag"] = "error"
        df.at[idx, "Valid_Repo_Step5"] = "no"
        df.at[idx, "removal_reason"] = "error"
        sleep(1)

    # 💾 Save progress every 10
    if i % 10 == 0:
        df.to_csv(interim_path, index=False)
        print(f"💾 Interim saved to: {interim_path}")

# Final cleanup and save
df.to_csv(output_path, index=False)
print(f"\n✅ Step 5 complete. Final output saved to: {output_path}")


C:\Users\gilla\AppData\Local\Temp\ipykernel_17140\2965020916.py:44: DtypeWarning: Columns (12) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(input_path)


🔍 Reviewing 24542 repos...

[1/24542] 🔍 Checking: ligi/gobandroid
[2/24542] 🔍 Checking: quran/quran_android
[3/24542] 🔍 Checking: facebook/facebook-android-sdk
[4/24542] 🔍 Checking: thunderbird/thunderbird-android
[5/24542] 🔍 Checking: wuan/bo-android
[6/24542] 🔍 Checking: UweTrottmann/SeriesGuide
[7/24542] 🔍 Checking: MikeOrtiz/TouchImageView
[8/24542] 🔍 Checking: bugsnag/bugsnag-android
[9/24542] 🔍 Checking: AndBible/and-bible
[10/24542] 🔍 Checking: rarnu/root-tools
💾 Interim saved to: C:\Android Mobile App\Step1_URL_Search\Type_2_Searching_Pipeline\step5_removal_keyword_check_output_interim.csv
[11/24542] 🔍 Checking: signalapp/Signal-Android
[12/24542] 🔍 Checking: andstatus/andstatus
[13/24542] 🔍 Checking: JetBrains/kotlin
[14/24542] 🔍 Checking: getsentry/sentry-java
[15/24542] 🔍 Checking: ubergeek42/weechat-android
[16/24542] 🔍 Checking: gentlecat/counter
[17/24542] 🔍 Checking: mtotschnig/MyExpenses
[18/24542] 🔍 Checking: persian-calendar/persian-calendar
[19/24542] 🔍 Checking: ano